In [0]:
# Mount ADLS Gen2
# Required each time the cluster is restarted which should be only on the first notebook as the
tiers = ["bronze", "silver", "gold"]
adls_paths = {tier: f"abfss://{tier}@datalukerimas.dfs.core.windows.net/" for tier in tiers}

# Accessing paths
bronze_adls = adls_paths["bronze"]
silver_adls = adls_paths["silver"]
gold_adls = adls_paths["gold"]

dbutils.fs.ls(bronze_adls)
dbutils.fs.ls(silver_adls)
dbutils.fs.ls(gold_adls)

[FileInfo(path='abfss://gold@datalukerimas.dfs.core.windows.net/earthquake_events_gold/', name='earthquake_events_gold/', size=0, modificationTime=1783395961000)]

In [0]:
import requests
import json
from datetime import date, timedelta

In [0]:
# # Remove this before running Data Factory Pipeline
# start_date = date.today() - timedelta(1)
# end_date = date.today()

# Data Factory
# Get base parameters
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")
start_date = dbutils.widgets.get("start_date")
end_date = dbutils.widgets.get("end_date")

In [0]:
# Construct the API URL with start and end dates provided by Data Factory, formatted for geojson output.
url = f"https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime={start_date}&endtime={end_date}"



In [0]:
try:
    # Make the GET request to fetch data
    response = requests.get(url)

    # Check if the request was successful
    response.raise_for_status()  # Raise HTTPError for bad responses (4xx or 5xx)
    data = response.json().get('features', [])

    if not data:
        print("No data returned for the specified date range.")
    else:
        # Specify the ADLS path
        file_path = f"{bronze_adls}/{start_date}_earthquake_data.json"

        # Save the JSON data
        json_data = json.dumps(data, indent=4)
        dbutils.fs.put(file_path, json_data, overwrite=True)
        print(f"Data successfully saved to {file_path}")

except requests.exceptions.RequestException as e:
    print(f"Error fetching data from API: {e}")


Wrote 260318 bytes.
Data successfully saved to abfss://bronze@datalukerimas.dfs.core.windows.net//2026-07-06_earthquake_data.json


In [0]:
data

[{'type': 'Feature',
  'properties': {'mag': 2.3,
   'place': '33 km SE of Seldovia, Alaska',
   'time': 1783382378372,
   'updated': 1783382540726,
   'tz': None,
   'url': 'https://earthquake.usgs.gov/earthquakes/eventpage/aka2026nhkbzo',
   'detail': 'https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=aka2026nhkbzo&format=geojson',
   'felt': None,
   'cdi': None,
   'mmi': None,
   'alert': None,
   'status': 'automatic',
   'tsunami': 0,
   'sig': 81,
   'net': 'ak',
   'code': 'a2026nhkbzo',
   'ids': ',aka2026nhkbzo,',
   'sources': ',ak,',
   'types': ',origin,phase-data,',
   'nst': 37,
   'dmin': 0.3,
   'rms': 0.8,
   'gap': 160,
   'magType': 'ml',
   'type': 'earthquake',
   'title': 'M 2.3 - 33 km SE of Seldovia, Alaska'},
  'geometry': {'type': 'Point', 'coordinates': [-151.258, 59.242, 0.1]},
  'id': 'aka2026nhkbzo'},
 {'type': 'Feature',
  'properties': {'mag': 1.7,
   'place': '5 km E of Naalehu, Hawaii',
   'time': 1783382234740,
   'updated': 1783382448740,
  

In [0]:
# Define your variables
output_data = {
    "start_date": start_date,
    "end_date": end_date,
    "bronze_adls": bronze_adls,
    "silver_adls": silver_adls,
    "gold_adls": gold_adls
}

# Serialize the dictionary to a JSON string
output_json = json.dumps(output_data)

# Log the serialized JSON for debugging
print(f"Serialized JSON: {output_json}")

# Return the JSON string
dbutils.notebook.exit(output_json)
